# Estimating pi by throwing darts

An ordinary Julia file: `include("montecarlo.jl")` runs it top to bottom.
Every notebook and manual page beside it is DERIVED from this one file by
`notebooks_from_source.jl` — purely from the `#~` marks you see below.

The label legend for this file (the five shipped labels, bound by this
project's own convention — the legend is data, not engine vocabulary):
  :label1 = parameters   :label2 = expensive   :label3 = solution
  :label4 = deep-dive    :label5 = private

<!-- #~ discard{ :label5 } -->

## Parameters

In [1]:
n_darts  = 10_000
rng_seed = 2026

2026

## The estimator
A dart lands uniformly in the unit square; the quarter-circle catches
pi/4 of them — so four times the hit fraction estimates pi:

```jldoctest
julia> round(4 * atan(1); digits = 4)   # the target the darts approach
3.1416
```

In [2]:
using Random
function estimate_pi(n; rng = MersenneTwister(rng_seed))
    hits = count(_ -> rand(rng)^2 + rand(rng)^2 <= 1, 1:n)
    4 * hits / n
end
estimate_pi(n_darts)

3.1148

## Exercise
Rewrite the estimator so it draws both coordinates in one pass and
measure the speedup.

In [3]:
function estimate_pi_fused(n; rng = MersenneTwister(rng_seed))
    hits = 0
    for _ in 1:n
        hits += ifelse(rand(rng)^2 + rand(rng)^2 <= 1, 1, 0)
    end
    4 * hits / n
end
estimate_pi_fused(n_darts)

3.1148

## Convergence study
More darts, better estimate — this cell is marked expensive: automated
executors that honor the `skip-execution` tag (nbclient, nbconvert) skip
it; an interactive Run All does not, so plain `include` and interactive
runs keep a fast default unless you opt in with the environment variable.

In [ ]:
n_many = get(ENV, "DARTS_FULL", "") == "" ? 100_000 : 100_000_000
estimate_pi(n_many)

## Why the error falls like one over sqrt of n

<!-- #~2 :label4 -->
The hit count is a Binomial sum, so the estimator's standard error is
`4 * sqrt(p * (1 - p) / n)` with `p = pi / 4`. Doubling the dart count
divides the error by sqrt of 2 — the check below is a doctest the
derived manual pages carry only where this deep-dive section survives:

```jldoctest
julia> round(4 * sqrt((pi / 4) * (1 - pi / 4) / 10_000); digits = 4)
0.0164
```

In [4]:
se(n) = 4 * sqrt((pi / 4) * (1 - pi / 4) / n)
se(10_000)

0.01642183367736324

<!-- #] -->
The section above closed; this line is deliberately detached from it.

---
*How this notebook was made:* derived from `notebooks/src/montecarlo.jl` by `notebooks_from_source.jl` (edition **full**; 0 cell(s) omitted by this edition's policy; 6 source line(s) removed and 0 hidden by the engine's own verdicts before any policy ran; verdict digest `eeaa844d`). Hidden lines and the source's `#~` metaLines travel INVISIBLY with this file — as HTML comments in markdown cells and under the `gometa` key in code-cell metadata — so the GoMeta data can be read and worked with in this artifact as in the source; discarded lines do not travel. Regenerate and byte-compare every generated file: `julia --startup-file=no --project=. notebooks_from_source.jl --check` — the executed edition is validated separately.